##Scoring and Inference

In [ ]:
!pip install bert_score rouge_score

import pandas as pd
import math
import numpy as np
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from collections import Counter

In [14]:
drug_results = pd.read_excel('results_LLM_scored.xlsx')
drug_results.head()

,Unnamed: 0,DDInterID_A,Drug_A,DDInterID_B,Drug_B,Level,Dataset,Category,question,answer,prompt,ground_truth,base_prediction,finetuned_prediction,base_score,finetuned_score
0,8266.0,DDInter20,Acetylsalicylic acid,DDInter347,Chamomile,Minor,A,Interaction involving alimentary tract and met...,How do Acetylsalicylic acid and Chamomile inte...,Acetylsalicylic acid and Chamomile are reporte...,\n Below is an instruction that describ...,Minor,Chamomile and Acetylsulaction have a Minor int...,Chamomile and AcetylSalicylic acid have a Mino...,5.0,7.0
1,65177.0,DDInter1037,Lepirudin,DDInter283,Cangrelor,Major,B,Interaction involving blood and blood forming ...,How do Lepirudin and Cangrelor interact?,Lepirudin and Cangrelor are reported to have a...,\n Below is an instruction that describ...,Major,Cangrelor and Lepirudine have a Moderate inter...,Cangrelors and Lepirudins have a Major interac...,3.0,8.0
2,175362.0,DDInter1789,Thalidomide,DDInter340,Ceritinib,Moderate,L,Interaction involving antineoplastic and immun...,How do Thalidomide and Ceritinib interact?,Thalidomide and Ceritinib are reported to have...,\n Below is an instruction that describ...,Moderate,Major drugs for a Moderate interaction for dis...,CeritinIB and Thalidomamide have a Moderate in...,5.0,8.0
3,212055.0,DDInter1109,Lyme disease vaccine (recombinant OspA),DDInter704,Everolimus,Moderate,L,Interaction involving antineoplastic and immun...,How do Lyme disease vaccine (recombinant OspA)...,Lyme disease vaccine (recombinant OspA) and Ev...,\n Below is an instruction that describ...,Moderate,Everolimus and Recombinant A. have a Moderate ...,Everolimus and Lyme disease vaccination (recom...,8.0,8.0
4,157736.0,DDInter8,Abiraterone,DDInter1032,Lefamulin,Major,L,Interaction involving antineoplastic and immun...,How do Abiraterone and Lefamulin interact?,Abiraterone and Lefamulin are reported to have...,\n Below is an instruction that describ...,Major,Major interaction for a Moderate interaction.,Lefamunin and AbiraterONE have a Major interac...,4.0,8.0


In [9]:
import numpy as np
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from collections import Counter

# ── Weights: 0.7(BERT) + 0.2(ROUGE-L) + 0.1(Token-F1)
WEIGHTS = {
    "bert":     0.700,
    "rouge_l":  0.1,
    "token_f1": 0.2,
}

_rouge_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# ── Core helpers
def _token_f1(ref: str, cand: str) -> float:
    r = Counter(ref.lower().split())
    c = Counter(cand.lower().split())
    common = sum((r & c).values())
    if common == 0:
        return 0.0
    precision = common / len(cand.split())
    recall    = common / len(ref.split())
    return 2 * precision * recall / (precision + recall)

def _rouge_l(ref: str, cand: str) -> float:
    return _rouge_scorer.score(ref, cand)['rougeL'].fmeasure

def _is_valid(val) -> bool:
    return isinstance(val, str) and val.strip() != ""

# ── Batched evaluation
def evaluate_pairs(base_answers, finetuned_answers) -> list[dict]:
    """
    Batched version — BERTScore is called ONCE on all valid pairs.
    ROUGE-L and Token-F1 are computed per-pair (cheap, no model load).
    """
    base_answers      = list(base_answers)
    finetuned_answers = list(finetuned_answers)

    assert len(base_answers) == len(finetuned_answers), \
        "Lists must be the same length."

    n           = len(base_answers)
    all_results = [None] * n

    # ── Separate valid vs skipped indices
    valid_idx = [
        i for i in range(n)
        if _is_valid(base_answers[i]) and _is_valid(finetuned_answers[i])
    ]
    skip_idx = [i for i in range(n) if i not in set(valid_idx)]

    print(f"Total pairs  : {n}")
    print(f"Valid pairs  : {len(valid_idx)}")
    print(f"Skipped pairs: {len(skip_idx)}")
    print("\nRunning BERTScore on all valid pairs (single batch)...")

    valid_base      = [base_answers[i].strip()      for i in valid_idx]
    valid_finetuned = [finetuned_answers[i].strip() for i in valid_idx]

    # ── Single BERTScore call for all valid pairs
    _, _, bert_f1s = bert_score(
    valid_finetuned, valid_base,
    model_type="distilbert-base-uncased",
    lang=None,
    verbose=True,
    batch_size=16
    )

    bert_f1s = bert_f1s.tolist()

    # ── ROUGE-L + Token-F1 per pair (fast)
    print("\nComputing ROUGE-L and Token-F1...")
    for j, i in enumerate(valid_idx):
        base      = valid_base[j]
        finetuned = valid_finetuned[j]

        bert  = bert_f1s[j]
        rouge = _rouge_l(base, finetuned)
        tf1   = _token_f1(base, finetuned)

        composite = (
            WEIGHTS["bert"]     * bert  +
            WEIGHTS["rouge_l"]  * rouge +
            WEIGHTS["token_f1"] * tf1
        )

        all_results[i] = {
            "index":     i,
            "bert_f1":   round(bert,      4),
            "rouge_l":   round(rouge,     4),
            "token_f1":  round(tf1,       4),
            "composite": round(composite, 4),
            "skipped":   False,
        }

    # ── Fill skipped rows
    for i in skip_idx:
        all_results[i] = {
            "index":     i,
            "bert_f1":   None,
            "rouge_l":   None,
            "token_f1":  None,
            "composite": None,
            "skipped":   True,
        }

    # ── Print table
    print(f"\n{'#':<5} {'BERT':>6} {'ROUGE-L':>8} {'TokF1':>7} {'Composite':>10}")
    print("─" * 42)
    for r in all_results:
        if r["skipped"]:
            print(f"{r['index']:<5} {'SKIPPED'}")
        else:
            print(f"{r['index']:<5} {r['bert_f1']:>6} {r['rouge_l']:>8} "
                  f"{r['token_f1']:>7} {r['composite']:>10}")

    # ── Summary
    valid = [r for r in all_results if not r["skipped"]]
    print(f"\n─── Averages ({len(valid)} valid, {len(skip_idx)} skipped) ───")
    for key in ["bert_f1", "rouge_l", "token_f1", "composite"]:
        avg = np.mean([r[key] for r in valid])
        print(f"  {key:<12}: {avg:.4f}")

    return all_results

In [10]:
# ── Usage

baseline_results = evaluate_pairs(drug_results['answer'], drug_results['base_prediction'])

Total pairs  : 4000
Valid pairs  : 3408
Skipped pairs: 592

Running BERTScore on all valid pairs (single batch)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/342 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/213 [00:00<?, ?it/s]

done in 403.43 seconds, 8.45 sentences/sec

Computing ROUGE-L and Token-F1...

#       BERT  ROUGE-L   TokF1  Composite
──────────────────────────────────────────
0     0.8539   0.2308  0.2692     0.6747
1     0.8579   0.1509  0.2264     0.6609
2     0.7711   0.1395  0.1395     0.5816
3     0.8525   0.2545  0.2909     0.6804
4      0.732    0.125  0.1667     0.5582
5     0.6357   0.0588     0.0     0.4509
6     0.6365   0.0588     0.0     0.4515
7     0.7788   0.2222  0.1778      0.603
8     SKIPPED
9     0.7697   0.2083  0.1667     0.5929
10     0.789   0.1463  0.1951     0.6059
11    0.7768   0.2273  0.2273     0.6119
12    0.8602   0.2553  0.3404     0.6958
13    0.7567   0.1739  0.1739     0.5819
14    0.8693   0.2264  0.2642     0.6839
15    0.6381   0.0606     0.0     0.4527
16    SKIPPED
17    0.7839   0.2222  0.2667     0.6243
18    0.7723   0.1277  0.1702     0.5875
19    0.8407   0.2222   0.254     0.6615
20    0.8521   0.3111  0.2667     0.6809
21    0.8383   0.1754  0.2807 

In [11]:
finetuned_results = evaluate_pairs(drug_results['answer'], drug_results['finetuned_prediction'])

Total pairs  : 4000
Valid pairs  : 4000
Skipped pairs: 0

Running BERTScore on all valid pairs (single batch)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/500 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/250 [00:00<?, ?it/s]

done in 607.89 seconds, 6.58 sentences/sec

Computing ROUGE-L and Token-F1...

#       BERT  ROUGE-L   TokF1  Composite
──────────────────────────────────────────
0     0.8798   0.3137  0.3922     0.7257
1     0.8727   0.3729  0.3051     0.7092
2     0.8879   0.3265  0.3673     0.7277
3      0.892   0.4407  0.4407     0.7566
4     0.8745   0.2857  0.3214      0.705
5     0.8918    0.383  0.4255     0.7476
6     0.8766    0.383  0.3404       0.72
7     0.8916   0.3265  0.3265     0.7221
8      0.881   0.3265  0.3673     0.7228
9     0.8885   0.3556  0.3556     0.7286
10    0.8911   0.3673  0.3673      0.734
11    0.8787    0.383  0.2979     0.7129
12    0.9048   0.3529  0.3922     0.7471
13    0.8742   0.3774  0.3396     0.7176
14    0.8747   0.3448  0.3103     0.7088
15    0.8666   0.3182  0.2727      0.693
16    0.8843   0.3478  0.3478     0.7233
17    0.8732   0.3265    0.28     0.6999
18    0.8907   0.5424  0.3793     0.7536
19    0.9114   0.4231  0.3774     0.7558
20    0.8585   0.